In [4]:
import glob
import os
import pandas as pd
import numpy as np

# --- Settings ---
serials = ['16024', '20654', '20655']   # Camera serials in desired order
csv_files = glob.glob('*.csv', recursive=True)  # All CSV files in directory
trials = sorted({os.path.basename(f)[:20] for f in csv_files})  # Unique trials detected

def DLC_trimmer(csv_file):
    """
    Clean DLC CSV by:
    - Dropping the first column (frame index)
    - Renaming columns using first two rows (scorer + bodypart)
    - Removing 'likelihood' columns
    """
    df = pd.read_csv(csv_file)
    df = df.drop(df.columns[0], axis=1)  # Drop first column by position
    df.columns = [f'{df.iloc[0, i]}_{df.iloc[1, i]}' for i in range(df.shape[1])]
    df = df.iloc[2:].reset_index(drop=True)  # Remove first 2 header rows
    return df.loc[:, ~df.columns.str.contains('likelihood')]  # Keep only x,y

# Mapping: (role, bodypart) → standardized pt labels
mapping = {
    ("chasee", "Head"): "pt1",
    ("chasee", "Abdomen"): "pt2",
    ("chaser", "Head"): "pt3",
    ("chaser", "Abdomen"): "pt4",
}

# --- Main processing loop ---
for trial in trials:
    # Find all files for this trial
    files = [f for f in csv_files if trial in os.path.basename(f)]
    if not files: 
        continue

    # Store processed DataFrames as {(serial, role): df}
    dfs = {}
    for f in files:
        base = os.path.basename(f)
        serial = next((s for s in serials if s in base), None)
        if not serial:
            print(f"Skipping {base}, no known serial found")
            continue
        role = "chasee" if "chasee" in base else "chaser"
        dfs[(serial, role)] = DLC_trimmer(f)

    # Build combined DataFrame with standardized point labels
    combined = pd.DataFrame()
    for (role, part), pt in mapping.items():
        for cam_idx, s in enumerate(serials, start=1):
            if (s, role) in dfs:
                df = dfs[(s, role)]
                combined[f"{pt}_cam{cam_idx}_X"] = df[f"{part}_x"]
                combined[f"{pt}_cam{cam_idx}_Y"] = df[f"{part}_y"]

    # Save cleaned & combined CSV
    out_file = f"{trial}_xypts.csv"
    combined.to_csv(out_file, index=False)
    print(f"Saved {out_file}, shape={combined.shape}")

    # --- Generate DLTdv-compatible supplemental CSVs ---

    n_frames = len(combined)

    # 1. offsets.csv
    offsets = pd.DataFrame(
        np.zeros((n_frames, len(serials))), 
        columns=[f"cam{i}_offset" for i in range(1, len(serials)+1)]
    )
    offsets.to_csv(f"{trial}_offsets.csv", index=False)
    print(f"Saved {trial}_offsets.csv, shape={offsets.shape}")

    # 2. xyzpts.csv (NaN-filled 3D coordinates)
    xyzpts_cols = []
    for i in range(1, 3):  # only pt1, pt2
        xyzpts_cols.extend([f"pt{i}_X", f"pt{i}_Y", f"pt{i}_Z"])
    xyzpts = pd.DataFrame(np.nan, index=range(n_frames), columns=xyzpts_cols)
    xyzpts.to_csv(f"{trial}_xyzpts.csv", index=False, na_rep='NaN')
    print(f"Saved {trial}_xyzpts.csv, shape={xyzpts.shape}")

    # 3. xyzres.csv (NaN-filled residuals)
    xyzres_cols = [f"pt{i}_dltres" for i in range(1, 3)]  # pt1, pt2
    xyzres = pd.DataFrame(np.nan, index=range(n_frames), columns=xyzres_cols)
    xyzres.to_csv(f"{trial}_xyzres.csv", index=False, na_rep='NaN')
    print(f"Saved {trial}_xyzres.csv, shape={xyzres.shape}")


Saved 2023-04-28_Trial1_1k_xypts.csv, shape=(2891, 24)
Saved 2023-04-28_Trial1_1k_offsets.csv, shape=(2891, 3)
Saved 2023-04-28_Trial1_1k_xyzpts.csv, shape=(2891, 6)
Saved 2023-04-28_Trial1_1k_xyzres.csv, shape=(2891, 2)
Saved 2023-04-28_Trial2_5k_xypts.csv, shape=(519, 24)
Saved 2023-04-28_Trial2_5k_offsets.csv, shape=(519, 3)
Saved 2023-04-28_Trial2_5k_xyzpts.csv, shape=(519, 6)
Saved 2023-04-28_Trial2_5k_xyzres.csv, shape=(519, 2)
Saved 2023-04-28_Trial3_5k_xypts.csv, shape=(766, 24)
Saved 2023-04-28_Trial3_5k_offsets.csv, shape=(766, 3)
Saved 2023-04-28_Trial3_5k_xyzpts.csv, shape=(766, 6)
Saved 2023-04-28_Trial3_5k_xyzres.csv, shape=(766, 2)
Saved 2023-04-28_Trial4_5k_xypts.csv, shape=(650, 24)
Saved 2023-04-28_Trial4_5k_offsets.csv, shape=(650, 3)
Saved 2023-04-28_Trial4_5k_xyzpts.csv, shape=(650, 6)
Saved 2023-04-28_Trial4_5k_xyzres.csv, shape=(650, 2)
Saved 2023-04-28_Trial5_5k_xypts.csv, shape=(850, 24)
Saved 2023-04-28_Trial5_5k_offsets.csv, shape=(850, 3)
Saved 2023-04-28_Tr